In [ ]:
# Run once per environment
!pip install numpy pandas h5py matplotlib seaborn ipywidgets ipympl

In [ ]:
file_path = '/path/to/your/file.hdf5'  # Replace with your actual file path

In [2]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import h5py
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import Button
from IPython.display import display, clear_output

# QALPA palette (same as img/qalpa_figure.py and img/navigator_figure.py)
QALPA = dict(ink="#1f2328", ink2="#52514e", grid="#e1e0d9", blue="#2a78d6", orange="#eb6834", aqua="#1baf7a")
QALPA_BLUES = LinearSegmentedColormap.from_list(
    "qalpa_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])   # one hue, light -> dark
QALPA_ORANGES = LinearSegmentedColormap.from_list("qalpa_oranges", ["#fde3d6", "#f6b08f", "#eb6834", "#a8431c"])
plt.rcParams.update({"axes.edgecolor": QALPA["grid"], "axes.labelcolor": QALPA["ink2"],
                     "xtick.color": QALPA["ink2"], "ytick.color": QALPA["ink2"], "grid.color": QALPA["grid"]})

### 3D trajectory

In [4]:
def extract_properties3(file_path, property1, property2, property3):
    """
    Extract arrays for 'HLgap' and 'eMBD' from the specified HDF5 file and dataset.

    Parameters:
    -----------
    file_path : str
        Path to the HDF5 file.
    property1 : str
        The first property to extract (e.g., 'HLgap').
    property2 : str
        The second property to extract (e.g., 'eMBD').
    property3 : str
        The third property to extract (e.g., 'atNUM').

    Returns:
    --------
    tuple
        Two lists containing the extracted values for the specified properties.
    """

    property1_values = []
    property2_values = []
    property3_values = []

    with h5py.File(file_path, "r") as f:
        for subset in f.keys():
            if isinstance(f[subset], h5py.Group):
                for geom in f[subset].keys():
                    if property1 in f[subset][geom].keys() and property2 in f[subset][geom].keys() and property3 in f[subset][geom].keys():
                        prop1 = f[subset][geom][property1][()]
                        prop2 = f[subset][geom][property2][()]
                        prop3 = f[subset][geom][property3][()]

                        # Convert to scalar if numpy array with zero dims
                        if hasattr(prop1, 'shape') and prop1.shape == ():
                            prop1 = float(prop1)
                        if hasattr(prop2, 'shape') and prop2.shape == ():
                            prop2 = float(prop2)
                        if hasattr(prop3, 'shape') and prop3.shape == ():
                            prop3 = float(prop3)

                        property1_values.append(prop1)
                        property2_values.append(prop2)
                        property3_values.append(prop3)

    return property1_values, property2_values, property3_values

In [ ]:
def _clean_property_values(name, values):
    """Convert 'atNUM' arrays to counts and flatten all values."""
    if name == 'atNUM':
        values = [len(v) for v in values]
    return np.array(values).flatten()

def show_properties3(
    x_name, x_values,
    y_name, y_values,
    c_name, c_values,
    gridsize=50,
    cmap=QALPA_BLUES,
    vmin=None,
    vmax=None
):
    """
    Interactive hexbin plot of x vs y coloured by the mean of c; left clicks add waypoints.

    Each click snaps to the nearest hexagon centre and records (x, y, mean c) of that hexagon.
    """
    global fig, ax, cid, line, clicked_points
    clicked_points = []  # reset each time
    x_values = _clean_property_values(x_name, x_values)
    y_values = _clean_property_values(y_name, y_values)
    c_values = _clean_property_values(c_name, c_values)

    fig, ax = plt.subplots(figsize=(8, 6))
    hb = ax.hexbin(
        x_values, y_values, C=c_values,
        gridsize=gridsize,
        mincnt=1,
        reduce_C_function=np.mean,
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        linewidths=0.2, edgecolors="white",
    )
    cb = plt.colorbar(hb, ax=ax, label=f"mean {c_name}")
    cb.outline.set_visible(False)
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f"{x_name} vs {y_name} coloured by {c_name}", color=QALPA["ink"], loc="left", fontweight="bold")
    ax.grid(True, linewidth=0.6)
    ax.set_axisbelow(True)

    # waypoints: orange line and markers with a white ring
    (line,) = ax.plot([], [], marker='o', color=QALPA["orange"], lw=2.2, markersize=8,
                      markeredgecolor="white", markeredgewidth=1.5, zorder=5)

    def on_click(event):
        if event.inaxes != ax or event.button != 1:  # left clicks only
            return
        offsets = hb.get_offsets()  # hexagon centres, shape (n_hex, 2)
        cvals = hb.get_array()      # reduced C values
        idx = np.argmin(np.hypot(offsets[:, 0] - event.xdata, offsets[:, 1] - event.ydata))
        clicked_points.append((float(offsets[idx, 0]), float(offsets[idx, 1]), float(cvals[idx])))
        line.set_data([p[0] for p in clicked_points], [p[1] for p in clicked_points])
        refresh_property_tables()   # keep one SAScore/QED field per waypoint
        fig.canvas.draw_idle()

    cid = fig.canvas.mpl_connect("button_press_event", on_click)
    plt.show()

In [ ]:
# KDE plots of an input vs an output (augmented) dataset
def show_kde(prop1_name, prop1_in_values, prop1_out_values, prop2_name, prop2_in_values, prop2_out_values,
             save_path=None):
    """
    Side-by-side 2D KDE plots of two properties for the input and output datasets, plus an overlay.
    save_path: optional file to save the figure to.
    """
    clean = lambda name, v: _clean_property_values(name, v)
    x_in, x_out = clean(prop1_name, prop1_in_values), clean(prop1_name, prop1_out_values)
    y_in, y_out = clean(prop2_name, prop2_in_values), clean(prop2_name, prop2_out_values)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.kdeplot(x=x_in, y=y_in, fill=True, cmap=QALPA_BLUES, thresh=0.05, levels=10, ax=axes[0])
    axes[0].set_title(f'Input data: {prop1_name} vs {prop2_name}')
    sns.kdeplot(x=x_out, y=y_out, fill=True, cmap=QALPA_ORANGES, thresh=0.05, levels=10, ax=axes[1])
    axes[1].set_title(f'Output data: {prop1_name} vs {prop2_name}')
    sns.kdeplot(x=x_in, y=y_in, fill=False, color=QALPA["blue"], thresh=0.05, levels=10, linewidths=1.5,
                ax=axes[2], label='Input')
    sns.kdeplot(x=x_out, y=y_out, fill=False, color=QALPA["orange"], thresh=0.05, levels=10, linewidths=1.5,
                ax=axes[2], label='Output')
    axes[2].set_title(f'Overlay: {prop1_name} vs {prop2_name}')
    axes[2].legend()
    for a in axes:
        a.set_xlabel(prop1_name)
        a.set_ylabel(prop2_name)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

In [ ]:
# --- Global state ---
clicked_points = []
fig = None
ax = None
cid = None
line = None

# Properties available in the initial dataset
molecule_prop = [
    'DIP', 'HLgap', 'atNUM', 'eAT', 'eC', 'eEE', 'eH',
    'eKIN', 'eL', 'eMBD', 'eNE', 'eNN', 'ePBE0', 'ePBE0+MBD',
    'eTS', 'eX', 'eXC', 'eXX', 'mC6', 'mPOL'
]

# Widgets
prop1_dropdown = widgets.Dropdown(options=molecule_prop, value='HLgap', description='X-axis Property:', style={'description_width': 'initial'})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, value='eMBD', description='Y-axis Property:', style={'description_width': 'initial'})
prop3_dropdown = widgets.Dropdown(options=molecule_prop, value='atNUM', description='Colour Property:', style={'description_width': 'initial'})
show_button = widgets.Button(description="Show Plot")
save_button = widgets.Button(description="Save Plot + CSV")
undo_btn = Button(description="Undo")
clear_btn = Button(description="Clear")
stop_btn = Button(description="Stop Picking")
output3 = widgets.Output()

# --- Optional per-waypoint score thresholds (only read with query.score_reference: trajectory) ---
sascore_btn = widgets.ToggleButton(value=False, description="Add SAScore")
qed_btn = widgets.ToggleButton(value=False, description="Add QED")
sascore_box = widgets.VBox([])
qed_box = widgets.VBox([])


def _score_values(box):
    return [w.value for w in box.children if isinstance(w, widgets.FloatText)]


def refresh_property_tables():
    """One editable SAScore/QED field per waypoint; values already typed are kept."""
    n = len(clicked_points)
    for active, box, label in ((sascore_btn.value, sascore_box, "SAScore values:"), (qed_btn.value, qed_box, "QED values:")):
        if not active:
            box.children = []
            continue
        old = _score_values(box)
        box.children = [widgets.Label(label)] + [widgets.FloatText(value=old[i] if i < len(old) else 0.0) for i in range(n)]


def save_trajectory():
    if fig is None or not clicked_points:
        with output3:
            print("No points to save: show the plot and click waypoints first.")
        return
    props = [prop1_dropdown.value, prop2_dropdown.value, prop3_dropdown.value]
    stem = f"trajectory_{props[0]}_{props[1]}_{props[2]}"
    fig.savefig(f"{stem}.png", dpi=300)

    df = pd.DataFrame(clicked_points, columns=props)
    if 'atNUM' in props:
        df['atNUM'] = df['atNUM'].round().astype(int)
    for active, box, column in ((sascore_btn.value, sascore_box, "SAScore"), (qed_btn.value, qed_box, "QED")):
        if active:
            values = _score_values(box)
            if len(values) != len(df):
                with output3:
                    print(f"{column}: {len(values)} values for {len(df)} waypoints; toggle the button off and on.")
                return
            df[column] = values   # column names match query.scores in experiment_config.yaml
    df.to_csv(f"{stem}.csv", index=False)
    with output3:
        print(f"Saved {len(df)} waypoints to {stem}.csv (copy it to Experiments/<experiment>/configs/)")


def on_show_plot(_):
    with output3:
        clear_output(wait=True)
        x_axis, y_axis, c_axis = prop1_dropdown.value, prop2_dropdown.value, prop3_dropdown.value
        x_values, y_values, c_values = extract_properties3(file_path, x_axis, y_axis, c_axis)
        show_properties3(x_axis, x_values, y_axis, y_values, c_axis, c_values)
        refresh_property_tables()

def _redraw():
    if line is not None:
        line.set_data([p[0] for p in clicked_points], [p[1] for p in clicked_points])
        fig.canvas.draw_idle()
    refresh_property_tables()

def on_undo(_):
    if clicked_points:
        clicked_points.pop()
        _redraw()

def on_clear(_):
    clicked_points.clear()
    _redraw()

def on_stop(_):
    global cid
    if fig is not None and cid is not None:
        fig.canvas.mpl_disconnect(cid)
        cid = None
        with output3:
            print("Click picking disabled.")

# Bind callbacks
show_button.on_click(on_show_plot)
save_button.on_click(lambda _: save_trajectory())
undo_btn.on_click(on_undo)
clear_btn.on_click(on_clear)
stop_btn.on_click(on_stop)
sascore_btn.observe(lambda _: refresh_property_tables(), 'value')
qed_btn.observe(lambda _: refresh_property_tables(), 'value')

display(widgets.VBox([
    prop1_dropdown,
    prop2_dropdown,
    prop3_dropdown,
    widgets.HBox([show_button, save_button]),
    widgets.HBox([undo_btn, clear_btn, stop_btn]),
    widgets.HBox([sascore_btn, qed_btn]),
    sascore_box, qed_box,
    output3
]))